# QA: Phase Error Correction Quality

Inspect downsampled (128×128×64) velocity data for every patient in the splits file.
For each patient, a mid-axial slice is shown at a single cardiac frame:

| Row | Content |
|-----|---------|
| 1   | 4D flow magnitude, 3D cine |
| 2   | Uncorrected vx, vy, vz |
| 3   | Corrected vx, vy, vz |
| 4   | Difference (corrected − uncorrected) per component |
| 5   | Predicted correction before polyfit (raw network output, per-timepoint) |
| 6   | Predicted correction after polyfit (time-independent) |

Rows 5–6 are only available for patients with inference results.

Images are saved to `./qa_pec_images/`.

In [13]:
import platform
from pathlib import Path

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import pandas as pd

from vascular_superenhancement.utils.path_config import load_path_config, _PROJECT_ROOT

config_name = "local_mac" if platform.system() == "Darwin" else "all_patients"
pc = load_path_config(config_name)

WORKING_DIR = pc.working_dir
PATIENT_DATA_DIR = WORKING_DIR / "patient_data"
REPO_ROOT = _PROJECT_ROOT
DOWNSAMPLED_FOLDER = "downsampled_full_fov_128x128x64"

RUN_NAME = "glowing-microwave_epoch-69"
INFERENCE_DIR = WORKING_DIR.parent / "inference" / RUN_NAME

OUTPUT_DIR = Path("qa_pec_images")
OUTPUT_DIR.mkdir(exist_ok=True)

FRAME_INDEX = 4

print(f"Patient data dir: {PATIENT_DATA_DIR}")
print(f"Inference dir:    {INFERENCE_DIR}")
print(f"Output dir:       {OUTPUT_DIR.resolve()}")

Patient data dir: /home/ayeluru/vascular-superenhancement-4d-flow/working_dir/all_patients/patient_data
Inference dir:    /home/ayeluru/vascular-superenhancement-4d-flow/working_dir/inference/glowing-microwave_epoch-69
Output dir:       /home/ayeluru/vascular-superenhancement-4d-flow/notebooks/data-qa/qa_pec_images


In [14]:
splits_df = pd.read_csv(REPO_ROOT / "splits" / "splits_01-15-26.csv")
patients_df = splits_df[splits_df["split"].isin(["train", "validation", "test"])].copy()
patients_df = patients_df.sort_values(["split", "patient_id"]).reset_index(drop=True)

print(f"Total patients to inspect: {len(patients_df)}")
print(patients_df["split"].value_counts())

Total patients to inspect: 215
split
train         165
test           28
validation     22
Name: count, dtype: int64


In [18]:
COMPONENTS = ["vx", "vy", "vz"]

MAG_TISSUE_PERCENTILE = 5


def load_axial_slice(nifti_path: Path, z_idx: int | None = None) -> np.ndarray:
    """Load a NIfTI file and return an axial slice as a 2-D array.

    If z_idx is None, the mid-axial slice is used.
    """
    vol = nib.load(str(nifti_path)).get_fdata(dtype=np.float32)
    if z_idx is None:
        z_idx = vol.shape[2] // 2
    return vol[:, :, z_idx]


def make_tissue_mask(mag_slice: np.ndarray) -> np.ndarray:
    """Threshold on magnitude to separate tissue from air."""
    thresh = np.percentile(mag_slice[mag_slice > 0], MAG_TISSUE_PERCENTILE)
    return mag_slice > thresh


def apply_mask(arr: np.ndarray, mask: np.ndarray) -> np.ma.MaskedArray:
    """Return a masked array where air pixels are masked out."""
    return np.ma.masked_where(~mask, arr)


def _try_load_axial_slice(nifti_path: Path, z_idx: int | None = None) -> np.ndarray | None:
    """Load an axial slice if the file exists, otherwise return None."""
    if not nifti_path.exists():
        return None
    return load_axial_slice(nifti_path, z_idx)


def make_qa_figure(pid: str, split: str, frame: int, ds_root: Path,
                   inf_root: Path | None = None,
                   z_idx: int | None = None) -> plt.Figure:
    """Generate a QA figure. If z_idx is None the mid-axial slice is used."""
    mag_path = ds_root / "4d_flow_mag" / f"4d_flow_mag_{pid}_frame_{frame:02d}.nii.gz"
    cine_path = ds_root / "3d_cine" / f"3d_cine_{pid}_frame_{frame:02d}.nii.gz"

    mag_slice = load_axial_slice(mag_path, z_idx)
    cine_slice = _try_load_axial_slice(cine_path, z_idx)

    uncorr = {}
    corr = {}
    for comp in COMPONENTS:
        uncorr[comp] = _try_load_axial_slice(
            ds_root / f"4d_flow_{comp}" / f"4d_flow_{comp}_{pid}_frame_{frame:02d}.nii.gz", z_idx
        )
        corr[comp] = _try_load_axial_slice(
            ds_root / f"4d_flow_{comp}_corr" / f"4d_flow_{comp}_corr_{pid}_frame_{frame:02d}.nii.gz", z_idx
        )

    has_uncorr = all(uncorr[c] is not None for c in COMPONENTS)
    has_corr = all(corr[c] is not None for c in COMPONENTS)
    has_diff = has_uncorr and has_corr
    diff = {}
    if has_diff:
        diff = {comp: corr[comp] - uncorr[comp] for comp in COMPONENTS}

    # Inference predictions (only if inf_root provided and exists)
    pred_raw = {}
    pred_poly = {}
    if inf_root is not None and inf_root.exists():
        # Poly volumes are at full resolution — scale z_idx proportionally
        poly_z = None
        if z_idx is not None:
            first_poly = inf_root / "predicted_corrected_velocity" / f"pred_correction_vx_{pid}.nii.gz"
            if first_poly.exists():
                poly_nz = nib.load(str(first_poly)).shape[2]
                ds_nz = nib.load(str(mag_path)).shape[2]
                poly_z = int(round(z_idx / (ds_nz - 1) * (poly_nz - 1)))

        for comp in COMPONENTS:
            pred_raw[comp] = _try_load_axial_slice(
                inf_root / "raw_predictions" / f"pred_correction_{comp}_t{frame:02d}.nii.gz", z_idx
            )
            pred_poly[comp] = _try_load_axial_slice(
                inf_root / "predicted_corrected_velocity" / f"pred_correction_{comp}_{pid}.nii.gz", poly_z
            )
    has_pred_raw = all(pred_raw.get(c) is not None for c in COMPONENTS)
    has_pred_poly = all(pred_poly.get(c) is not None for c in COMPONENTS)

    tissue = make_tissue_mask(mag_slice)

    vel_max = 1.0
    if has_uncorr or has_corr:
        tissue_vals = []
        for c in COMPONENTS:
            if uncorr[c] is not None:
                tissue_vals.append(uncorr[c][tissue])
            if corr[c] is not None:
                tissue_vals.append(corr[c][tissue])
        if tissue_vals:
            vel_max = np.percentile(np.abs(np.concatenate(tissue_vals)), 99)

    diff_max = 1.0
    if has_diff:
        diff_vals = np.concatenate([diff[c][tissue] for c in COMPONENTS])
        diff_max = np.percentile(np.abs(diff_vals), 99)
        if diff_max == 0:
            diff_max = 1.0

    # Correction scale (shared for rows 4-5, based on tissue of the raw pred)
    corr_max = 1.0
    corr_tissue_vals = []
    if has_pred_raw:
        for c in COMPONENTS:
            corr_tissue_vals.append(pred_raw[c][tissue])
    if has_pred_poly:
        # poly is at different resolution; use all voxels for scale
        for c in COMPONENTS:
            corr_tissue_vals.append(pred_poly[c].ravel())
    if corr_tissue_vals:
        corr_max = np.percentile(np.abs(np.concatenate(corr_tissue_vals)), 99)
        if corr_max == 0:
            corr_max = 1.0

    bg_color = "0.25"
    vel_cmap = plt.cm.RdBu_r.copy()
    vel_cmap.set_bad(bg_color)

    z_label = f"z={z_idx}" if z_idx is not None else "z=mid"

    n_rows = 6
    fig, axes = plt.subplots(n_rows, 3, figsize=(14, n_rows * 4), constrained_layout=True)
    fig.suptitle(f"{pid}  [{split}]  frame {frame:02d}  {z_label}", fontsize=16, fontweight="bold")

    # Row 0: mag and cine
    axes[0, 0].imshow(mag_slice.T, origin="upper", cmap="gray")
    axes[0, 0].set_title("4D Flow Magnitude")
    if cine_slice is not None:
        axes[0, 1].imshow(cine_slice.T, origin="upper", cmap="gray")
        axes[0, 1].set_title("3D Cine")
    else:
        axes[0, 1].text(0.5, 0.5, "3D Cine\nnot available", transform=axes[0, 1].transAxes,
                         ha="center", va="center", fontsize=12, color="white")
        axes[0, 1].set_title("3D Cine (missing)")
    axes[0, 2].axis("off")

    # Row 1: uncorrected velocity
    for j, comp in enumerate(COMPONENTS):
        if uncorr[comp] is not None:
            im = axes[1, j].imshow(
                apply_mask(uncorr[comp], tissue).T,
                origin="upper", cmap=vel_cmap, vmin=-vel_max, vmax=vel_max,
            )
            axes[1, j].set_title(f"Uncorrected {comp}")
        else:
            axes[1, j].text(0.5, 0.5, f"Uncorr {comp}\nnot available", transform=axes[1, j].transAxes,
                             ha="center", va="center", fontsize=12, color="white")
            axes[1, j].set_title(f"Uncorrected {comp} (missing)")
    if has_uncorr:
        fig.colorbar(im, ax=axes[1, :].tolist(), fraction=0.02, pad=0.02, label="velocity")

    # Row 2: corrected velocity
    for j, comp in enumerate(COMPONENTS):
        if corr[comp] is not None:
            im = axes[2, j].imshow(
                apply_mask(corr[comp], tissue).T,
                origin="upper", cmap=vel_cmap, vmin=-vel_max, vmax=vel_max,
            )
            axes[2, j].set_title(f"Corrected {comp}")
        else:
            axes[2, j].text(0.5, 0.5, f"Corr {comp}\nnot available", transform=axes[2, j].transAxes,
                             ha="center", va="center", fontsize=12, color="white")
            axes[2, j].set_title(f"Corrected {comp} (missing)")
    if has_corr:
        fig.colorbar(im, ax=axes[2, :].tolist(), fraction=0.02, pad=0.02, label="velocity")

    # Row 3: difference (corrected - uncorrected)
    for j, comp in enumerate(COMPONENTS):
        if has_diff:
            im = axes[3, j].imshow(
                apply_mask(diff[comp], tissue).T,
                origin="upper", cmap=vel_cmap, vmin=-diff_max, vmax=diff_max,
            )
            axes[3, j].set_title(f"Diff {comp} (corr − uncorr)")
        else:
            axes[3, j].text(0.5, 0.5, f"Diff {comp}\nnot available", transform=axes[3, j].transAxes,
                             ha="center", va="center", fontsize=12, color="white")
            axes[3, j].set_title(f"Diff {comp} (missing)")
    if has_diff:
        fig.colorbar(im, ax=axes[3, :].tolist(), fraction=0.02, pad=0.02, label="Δ velocity")

    # Row 4: predicted correction before polyfit (raw network output)
    for j, comp in enumerate(COMPONENTS):
        if has_pred_raw:
            im = axes[4, j].imshow(
                apply_mask(pred_raw[comp], tissue).T,
                origin="upper", cmap=vel_cmap, vmin=-corr_max, vmax=corr_max,
            )
            axes[4, j].set_title(f"Pred Raw {comp}")
        else:
            axes[4, j].text(0.5, 0.5, f"Pred Raw {comp}\nnot available", transform=axes[4, j].transAxes,
                             ha="center", va="center", fontsize=12, color="white")
            axes[4, j].set_title(f"Pred Raw {comp} (missing)")
    if has_pred_raw:
        fig.colorbar(im, ax=axes[4, :].tolist(), fraction=0.02, pad=0.02, label="correction")

    # Row 5: predicted correction after polyfit (time-independent)
    for j, comp in enumerate(COMPONENTS):
        if has_pred_poly:
            im = axes[5, j].imshow(
                pred_poly[comp].T,
                origin="upper", cmap=vel_cmap, vmin=-corr_max, vmax=corr_max,
            )
            axes[5, j].set_title(f"Pred Polyfit {comp}")
        else:
            axes[5, j].text(0.5, 0.5, f"Pred Polyfit {comp}\nnot available", transform=axes[5, j].transAxes,
                             ha="center", va="center", fontsize=12, color="white")
            axes[5, j].set_title(f"Pred Polyfit {comp} (missing)")
    if has_pred_poly:
        fig.colorbar(im, ax=axes[5, :].tolist(), fraction=0.02, pad=0.02, label="correction")

    for ax in axes.ravel():
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_facecolor(bg_color)

    return fig

In [16]:
errors = []

for idx, row in patients_df.iterrows():
    pid = row["patient_id"]
    split = row["split"]
    ds_root = PATIENT_DATA_DIR / pid / "nifti" / DOWNSAMPLED_FOLDER

    if not ds_root.exists():
        print(f"[SKIP] {pid}: downsampled dir not found")
        errors.append((pid, split, "missing_dir"))
        continue

    n_mag_frames = len(list((ds_root / "4d_flow_mag").glob("*.nii.gz")))
    frame = min(FRAME_INDEX, n_mag_frames - 1)

    inf_root = INFERENCE_DIR / pid

    try:
        fig = make_qa_figure(pid, split, frame, ds_root, inf_root=inf_root)
        out_path = OUTPUT_DIR / f"{split}_{pid}.png"
        fig.savefig(out_path, dpi=120, bbox_inches="tight")
        plt.close(fig)
        print(f"[OK]   {pid} ({split}) -> {out_path.name}")
    except Exception as e:
        print(f"[ERR]  {pid} ({split}): {e}")
        errors.append((pid, split, str(e)))

print(f"\nDone. {len(patients_df) - len(errors)} succeeded, {len(errors)} errors.")
if errors:
    print("Errors:")
    for pid, split, msg in errors:
        print(f"  {pid} ({split}): {msg}")

[OK]   Balboloop (test) -> test_Balboloop.png
[OK]   Biswifo (test) -> test_Biswifo.png
[OK]   Bomatog (test) -> test_Bomatog.png
[OK]   Boochuto (test) -> test_Boochuto.png
[OK]   Boumorim (test) -> test_Boumorim.png
[OK]   Bovutou (test) -> test_Bovutou.png
[OK]   Cadotueg (test) -> test_Cadotueg.png
[OK]   Detodu (test) -> test_Detodu.png
[OK]   Diecudey (test) -> test_Diecudey.png
[OK]   Diepami (test) -> test_Diepami.png
[OK]   Diequipi (test) -> test_Diequipi.png
[OK]   Dithigog (test) -> test_Dithigog.png
[OK]   Dublafer (test) -> test_Dublafer.png
[OK]   Dujomal (test) -> test_Dujomal.png
[OK]   Elagieg (test) -> test_Elagieg.png
[OK]   Golotag (test) -> test_Golotag.png
[OK]   Grequafie (test) -> test_Grequafie.png
[OK]   Gueshifa (test) -> test_Gueshifa.png
[OK]   Kuquelok (test) -> test_Kuquelok.png
[OK]   Oduskueb (test) -> test_Oduskueb.png
[OK]   Quetode (test) -> test_Quetode.png
[OK]   Runusath (test) -> test_Runusath.png
[OK]   Sepigoo (test) -> test_Sepigoo.png
[OK]  

## Multi-slice deep-dive: Biswifo

In [19]:
DEEP_DIVE_PID = "Biswifo"
DEEP_DIVE_SPLIT = splits_df.loc[splits_df["patient_id"] == DEEP_DIVE_PID, "split"].iloc[0]

ds_root = PATIENT_DATA_DIR / DEEP_DIVE_PID / "nifti" / DOWNSAMPLED_FOLDER
inf_root = INFERENCE_DIR / DEEP_DIVE_PID

n_mag_frames = len(list((ds_root / "4d_flow_mag").glob("*.nii.gz")))
frame = min(FRAME_INDEX, n_mag_frames - 1)

ref_vol = nib.load(str(ds_root / "4d_flow_mag" / f"4d_flow_mag_{DEEP_DIVE_PID}_frame_{frame:02d}.nii.gz"))
n_z = ref_vol.shape[2]

N_SLICES = 16
slice_indices = np.linspace(0, n_z - 1, N_SLICES, dtype=int)

patient_dir = OUTPUT_DIR / DEEP_DIVE_PID
patient_dir.mkdir(exist_ok=True)

print(f"Patient: {DEEP_DIVE_PID} ({DEEP_DIVE_SPLIT}), frame={frame}, volume z-dim={n_z}")
print(f"Generating {N_SLICES} slices: {slice_indices.tolist()}")

for z in slice_indices:
    fig = make_qa_figure(
        DEEP_DIVE_PID, DEEP_DIVE_SPLIT, frame, ds_root,
        inf_root=inf_root, z_idx=int(z),
    )
    out_path = patient_dir / f"z{z:02d}.png"
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"  [OK] z={z:02d} -> {out_path.name}")

print(f"\nDone. Images saved to {patient_dir.resolve()}")

Patient: Biswifo (test), frame=4, volume z-dim=64
Generating 16 slices: [0, 4, 8, 12, 16, 21, 25, 29, 33, 37, 42, 46, 50, 54, 58, 63]
  [OK] z=00 -> z00.png
  [OK] z=04 -> z04.png
  [OK] z=08 -> z08.png
  [OK] z=12 -> z12.png
  [OK] z=16 -> z16.png
  [OK] z=21 -> z21.png
  [OK] z=25 -> z25.png
  [OK] z=29 -> z29.png
  [OK] z=33 -> z33.png
  [OK] z=37 -> z37.png
  [OK] z=42 -> z42.png
  [OK] z=46 -> z46.png
  [OK] z=50 -> z50.png
  [OK] z=54 -> z54.png
  [OK] z=58 -> z58.png
  [OK] z=63 -> z63.png

Done. Images saved to /home/ayeluru/vascular-superenhancement-4d-flow/notebooks/data-qa/qa_pec_images/Biswifo
